### Random Forest Regressor
[RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

In [30]:
import pandas as pd

import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score

In [4]:
DATA = Path('datasets')
concrete_data = pd.read_csv(DATA / "concrete_data.csv")

In [5]:
X = concrete_data.drop(columns='csMPa')
y = concrete_data['csMPa']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
rnd_reg = RandomForestRegressor(
    n_estimators=600,
    min_samples_leaf=1,
    max_leaf_nodes=12,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

rnd_reg.fit(X_train, y_train)
y_pred = rnd_reg.predict(X_test)

r2_score(y_test, y_pred)

0.7069618963223071

In [11]:
important_features = [
    pd.DataFrame({
        'feature': X.columns,
        'importance': rnd_reg.feature_importances_
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
]

In [12]:
important_features

[            feature  importance
 0               age    0.388671
 1            cement    0.370082
 2             water    0.124968
 3              slag    0.056795
 4  superplasticizer    0.038590
 5            flyash    0.010033
 6     fineaggregate    0.007724
 7   coarseaggregate    0.003138]

### BaggingRegressor + DecisionTreeRegressor (with splitter='random') = RandomForestRegressor

In [13]:
base_tree = DecisionTreeRegressor(
    splitter='random',
    max_leaf_nodes=12,
    random_state=42,
    ccp_alpha=0.0
)

bag_reg = BaggingRegressor(
    estimator=base_tree,
    n_estimators=600,
    bootstrap=True,
    max_samples=1.0,
    n_jobs=-1,
    random_state=42,
)

bag_reg.fit(X_train,y_train)
y_pred = bag_reg.predict(X_test)
r2_score(y_test, y_pred)

0.7016524988362387

In [27]:
importances = np.mean(
    [tree.feature_importances_ for tree in bag_reg.estimators_],
    axis=0
)



In [28]:
important_features = [
    pd.DataFrame({
        'feature': X.columns,
        'importance': importances
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
]

In [29]:
important_features

[            feature  importance
 0               age    0.358873
 1            cement    0.323365
 2             water    0.097615
 3  superplasticizer    0.086113
 4              slag    0.066314
 5            flyash    0.041394
 6     fineaggregate    0.017917
 7   coarseaggregate    0.008409]

### ExtraTreesRegressor
[Link](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html)

In [19]:
extra_reg = ExtraTreesRegressor(
    n_estimators=600,
    max_leaf_nodes=12,
    min_samples_leaf=1,
    max_features=1.0,
    n_jobs=-1,
    random_state=42,
)

extra_reg.fit(X_train, y_train)
y_pred = extra_reg.predict(X_test)

In [20]:
important_features = [
    pd.DataFrame({
        'feature': X.columns,
        'importance': extra_reg.feature_importances_
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
]
important_features

[            feature  importance
 0               age    0.373436
 1            cement    0.325687
 2  superplasticizer    0.090474
 3             water    0.084409
 4              slag    0.061295
 5            flyash    0.043604
 6     fineaggregate    0.016348
 7   coarseaggregate    0.004745]